# Détection d'attaques par Force Brute (Python & Power BI) by MilhaneAI

## Introduction
Dans le cadre de la surveillance d'un système d'information, l'analyse des logs de connexion est la première ligne de défense pour identifier des comportements malveillants. L'objectif de ce projet est de mener une investigation complète sur des tentatives d'intrusions, puis de créer des outils d'alerte

Ce projet s'articule autour de deux grands chapitres :

**Chapitre 1** : Investigation (Python/Pandas). Repérage d'une attaque bruteforce.

**Chapitre 2** : Automatisation de l'outil et utilisation de Power BI. Création d'un tableau de bord permettant de visualiser les menaces.


# Chapitre 1 : Traitement de la donnée et détection (Python)
## Importation des librairies et lecture du dataset


In [110]:
import pandas as pd
df = pd.read_csv("/Users/milhane/PyCharmMiscProject/Projet Cyber 1/logs_securite_bruts.csv")
df.head(10)

,Date_Heure,Utilisateur,Adresse_IP,Localisation,Action,Statut
0,2026-03-01 08:24:00,employe_019,192.168.1.192,"Lyon, France",CONNEXION,ECHEC
1,2026-03-01 08:27:00,employe_018,192.168.1.139,"Paris, France",Connexion,succes
2,2026-03-01 08:29:00,employe_014,192.168.1.25,"Lyon, France",Connexion,succes
3,2026-03-01 08:49:00,employe_002,192.168.1.56,"Lyon, France",Connexion,NaN
4,2026-03-01 08:58:00,employe_015,192.168.1.41,"Grenoble, France",CONNEXION,ECHEC
5,2026-03-01 09:00:00,employe_014,192.168.1.45,"Lyon, France",CONNEXION,ECHEC
6,2026-03-01 09:20:00,employe_014,192.168.1.156,"Grenoble, France",telechargement,Echec
7,2026-03-01 09:36:00,employe_002,192.168.1.235,"Paris, France",CONNEXION,Echec
8,2026-03-01 09:43:00,employe_003,192.168.1.117,"Lyon, France",Connexion,succes
9,2026-03-01 09:55:00,employe_005,192.168.1.9,"Grenoble, France",Connexion,Succes


## Data Cleaning
On va dans un premier temps uniformiser le noms des "Actions" et des "Statut"

In [111]:
df['Action'] = df['Action'].str.strip().str.lower()
df['Statut'] = df['Statut'].str.strip().str.lower()


Puis supprimer les valeurs vides/erronées en les remplaçant par "inconnu".

In [112]:
df['Adresse_IP'] = df['Adresse_IP'].fillna('inconnu')
df['Statut'] = df['Statut'].fillna('inconnu')

## Parsing des dates
De sorte à pouvoir mieux manipuler les dates par la suite, on peut les parser pour que le format bien reconnu. Sans oublier de faire un petit test pour vérfier le bon format de données.

In [128]:
df['Date_Heure'] = pd.to_datetime(df['Date_Heure'])

In [129]:

print(df.dtypes)

Date_Heure      datetime64[us]
Utilisateur                str
Adresse_IP                 str
Localisation               str
Action                     str
Statut                     str
dtype: object


In [130]:
print(df['Action'].value_counts())

Action
connexion         959
telechargement    301
deconnexion       300
Name: count, dtype: int64


## Détection du bruteforce de l'utilisateur
Le bruteforce désigne une attaque qui consiste à tester à plusieurs reprises des mots de passe afin de se connecter à un service donné.
Ainsi pour identifier les utilisateurs qui ont essayé à plusieurs reprises de se connecter, on va observer quand le "Statut" a mené à un "connexion" et que "Action" donne "echec". En clair, on se pose la question suivante : quels sont les utilisateurs qui ont tenté de se connecter sans succès.

In [140]:
df_echecs = df[ (df['Statut'] == 'echec') & (df['Action'] == 'connexion') ]
df_echecs['Utilisateur'].value_counts()

Utilisateur
employe_004    70
employe_018    28
employe_015    24
employe_009    24
employe_011    23
employe_017    22
employe_014    20
employe_008    19
employe_012    17
employe_005    17
employe_020    16
employe_013    16
employe_001    15
employe_016    15
employe_007    15
employe_019    14
employe_006    12
employe_002    11
employe_010    11
employe_003     8
Name: count, dtype: int64

Ce qui ressort assez rapidement c'est que "employe_004" a essayé de se connecter à 70 reprises, ce qui est relativement élevé. Plus de deux fois plus que "employe_018", ce qui éveille évidemment les soupçons.

On va donc naturellement s'intéresser à celui-ci :

In [141]:
df_suspect = df_echecs[df_echecs['Utilisateur'] == 'employe_004']

Malheureusement (ou heureusement) se connecter à grand nombre de reprises ne suffit pas à qualifier les intentions de quelqu'un. Peut-être s'agit-il d'une personne qui dispose d'une mémoire un peu vieillissante et qui lui fait défaut.

On va donc s'intéresser à la fréquence des connexions. Quand a lieu la première et la dernière. Car 70 tentatives sur deux semaines ce n'est pas problématique. En revanche sur 3 jours, il y a de quoi se poser des questions...

In [142]:
duree = df_suspect['Date_Heure'].max() - df_suspect['Date_Heure'].min()
nombre_tentatives = len(df_suspect)

print(f"L'employe_004 a réalisé {nombre_tentatives} tentatives ratées.")
print(f"La durée totale de cette attaque est de : {duree}")

L'employe_004 a réalisé 70 tentatives ratées.
La durée totale de cette attaque est de : 13 days 06:16:00


13 jours... ce n'est pas concluant. Il est tout à fait possible que l'employé utilisait convenablement son compte sur une certaine plage. Puis que par la suite, un pirate a tenté de s'attaquer à lui. Dès lors il faudrait plutôt raisonner par fréquence journalière.

In [149]:
df_suspect = df_suspect.sort_values(by='Date_Heure', ascending=False)
df_suspect.head(10)

,Date_Heure,Utilisateur,Adresse_IP,Localisation,Action,Statut,Ecart_Temps
1500,2026-03-14 16:42:00,employe_004,192.168.1.20,"Lyon, France",connexion,echec,0 days 01:51:00
1485,2026-03-14 14:51:00,employe_004,192.168.1.221,"Grenoble, France",connexion,echec,0 days 16:51:00
1411,2026-03-13 22:00:00,employe_004,192.168.1.225,"Lyon, France",connexion,echec,0 days 14:37:00
1343,2026-03-13 07:23:00,employe_004,192.168.1.106,"Grenoble, France",connexion,echec,1 days 06:47:00
1218,2026-03-12 00:36:00,employe_004,192.168.1.109,"Paris, France",connexion,echec,0 days 05:48:00
1189,2026-03-11 18:48:00,employe_004,192.168.1.226,"Lyon, France",connexion,echec,0 days 17:22:00
1114,2026-03-11 01:26:00,employe_004,192.168.1.67,"Grenoble, France",connexion,echec,0 days 09:58:00
1068,2026-03-10 15:28:00,employe_004,192.168.1.104,"Grenoble, France",connexion,echec,0 days 04:19:00
1043,2026-03-10 11:09:00,employe_004,192.168.1.238,"Lyon, France",connexion,echec,0 days 13:57:00
955,2026-03-09 21:12:00,employe_004,192.168.1.232,"Grenoble, France",connexion,echec,0 days 01:42:00


In [144]:
df_suspect.groupby(df_suspect['Date_Heure'].dt.date).size()

Date_Heure
2026-03-01     2
2026-03-03     3
2026-03-05    47
2026-03-06     1
2026-03-07     1
2026-03-08     2
2026-03-09     5
2026-03-10     2
2026-03-11     2
2026-03-12     1
2026-03-13     2
2026-03-14     2
dtype: int64

Ca saute aux yeux ! Il y a eu plus de 47 connexions le 5 mars. Il y n'y aucun doute désormais. Mais il faut maintenant ce questionner sur le fait que le pirate a réussi ou non à se connecter après ses tentatives. Dans quel cas, ce serait terrible car cela viendrait à compromettre la sécurité entière de l'organisation.

In [145]:
connexions_reussies = df[(df['Utilisateur'] == 'employe_004') & (df['Statut'] == 'Succès')]
connexions_reussies

,Date_Heure,Utilisateur,Adresse_IP,Localisation,Action,Statut


C'est une bonne nouvelle. Il n'a pas réussi à se connecter !

On peut aussi se poser d'autres questions afin d'étayer notre analyse et faire signaler avec le plus de détails possibles l'attaque. Comme en regardant le temps écoulé entre chaque connexion. Un bot ayant essayé de cracker le code peut réaliser un grand nombre de combinaisons en un court laps de temps.

In [150]:
df_suspect = df_suspect.sort_values(by='Date_Heure', ascending=True)
df_suspect['Ecart_Temps'] = df_suspect['Date_Heure'].diff()
df_suspect.head(10)

,Date_Heure,Utilisateur,Adresse_IP,Localisation,Action,Statut,Ecart_Temps
10,2026-03-01 10:26:00,employe_004,192.168.1.108,"Grenoble, France",connexion,echec,NaT
28,2026-03-01 14:01:00,employe_004,192.168.1.6,"Paris, France",connexion,echec,0 days 03:35:00
224,2026-03-03 14:17:00,employe_004,192.168.1.248,"Paris, France",connexion,echec,2 days 00:16:00
234,2026-03-03 16:22:00,employe_004,192.168.1.14,"Lyon, France",connexion,echec,0 days 02:05:00
269,2026-03-03 23:26:00,employe_004,192.168.1.252,"Paris, France",connexion,echec,0 days 07:04:00
407,2026-03-05 08:05:00,employe_004,192.168.1.8,"Paris, France",connexion,echec,1 days 08:39:00
425,2026-03-05 12:21:00,employe_004,192.168.1.3,"Paris, France",connexion,echec,0 days 04:16:00
432,2026-03-05 14:00:00,employe_004,185.15.56.22,"Moscou, Russie",connexion,echec,0 days 01:39:00
433,2026-03-05 14:00:02,employe_004,185.15.56.22,"Moscou, Russie",connexion,echec,0 days 00:00:02
434,2026-03-05 14:00:04,employe_004,185.15.56.22,"Moscou, Russie",connexion,echec,0 days 00:00:02


On peut maintenant définir un seuil de vigilance. À partir de combien de secondes entre chaque connexion, c'est suspect ?! Prenons par exemple 3.

In [147]:
df_alertes = df_suspect[ df_suspect['Ecart_Temps'] < pd.Timedelta(seconds=3) ]
jours_problematiques = df_alertes.groupby(df_alertes['Date_Heure'].dt.date).size()
print(jours_problematiques)

Date_Heure
2026-03-05    44
dtype: int64


Notre fameux seuil de 3 secondes a été dépassé à 44 reprises toujours le 5 mars.

Toujours dans cette logique de reporting, on peut s'intéresser aux IPs qui se sont connectées. Si elles sont différentes ça laisse supposer que l'attaquant essayé de bypass les sécurités type "3 mots de passe restants" en utilisant plusieurs machines différentes (avec un bot.

In [148]:
ip_problematiques = df_alertes['Adresse_IP'].value_counts()
print(ip_problematiques)

Adresse_IP
185.15.56.22    44
Name: count, dtype: int64


Ainsi l'attaquant a utilisé qu'une seule machine.

# Bilan de l'investigation et Préconisations

L'analyse des logs a permis de caractériser une attaque par force brute avérée et automatisée, ciblant spécifiquement le compte employe_004. Bien que l'investigation prouve que l'attaque n'a pas abouti (absence de connexion réussie post-incident pour cet utilisateur), cet événement met en évidence la nécessité de durcir les politiques d'accès.

Pour sécuriser l'infrastructure et bloquer de futures tentatives similaires, les plans d'action suivants sont recommandés :

- **Politique de verrouillage de compte** : Pour bloquer temporairement un compte utilisateur après 3 à 5 tentatives de connexion échouées consécutives.

- **Limitation de débit (Rate Limiting)** : Imposer un délai technique obligatoire (ex: 3 secondes) entre chaque tentative de saisie de mot de passe afin de neutraliser l'efficacité des scripts automatisés.

- **Principe de précaution** : Forcer la réinitialisation du mot de passe de l'utilisateur ciblé et vérifier la robustesse de la politique globale des mots de passe.